# 04 — RAG Pipeline (Retrieval-Augmented Generation)
## Helpdesk IA — GEISER | PFE 2026

### Objectif
Construire un système RAG capable de :
- **Indexer** les tickets résolus dans une base vectorielle (ChromaDB)
- **Rechercher** les tickets similaires par similarité sémantique (cosinus)
- **Suggérer** des réponses basées sur les cas résolus dans le passé

### Architecture
```
Nouveau ticket → Embedding → ChromaDB → Tickets similaires → Suggestion
```

### Stack technique
- **ChromaDB** — base vectorielle persistante (locale)
- **Sentence Transformers** — embeddings multilingues (français ✅)
- **Similarité cosinus** — mesure de similarité sémantique

### Dataset
- **Source** : `dataset_labeled.csv` (produit par notebook 01)
- **Tickets indexés** : tickets résolus et fermés uniquement (base de connaissance)

---
## 1. Installation des dépendances

In [ ]:
# ============================================================
# INSTALLATION DÉPENDANCES
# ============================================================
import subprocess
import sys

packages = [
    'langchain',
    'langchain-community',
    'langchain-chroma',
    'chromadb',
    'sentence-transformers',
    'langchain-huggingface',
]

for pkg in packages:
    print(f"Installation {pkg}...")
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', pkg, '-q'
    ])

print("\n✅ Toutes les dépendances installées !")

---
## 2. Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import os
import re
import warnings
warnings.filterwarnings('ignore')

# LangChain
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

print("✅ Imports OK")

---
## 3. Chargement des Données

On charge les tickets **résolus et fermés** uniquement
car ils ont une solution connue → base de connaissance.

In [ ]:
# ============================================================
# CHARGEMENT TICKETS RÉSOLUS
# ============================================================
df = pd.read_csv('../../data/dataset_labeled.csv', encoding='utf-8')

# Garder uniquement tickets résolus/fermés
df_resolved = df[df['statut'].isin(['Fermé', 'Résolu'])].copy()

print(f"Total tickets     : {len(df)}")
print(f"Tickets résolus   : {len(df_resolved)}")
print(f"Tickets ouverts   : {len(df[df['statut'] == 'Ouvert'])}")

print(f"\nDistribution statuts :")
print(df['statut'].value_counts())

df_resolved.head(3)

---
## 4. Préparation des Documents

Chaque ticket résolu devient un **Document** LangChain
avec son texte et ses métadonnées.

In [ ]:
# ============================================================
# PRÉPARATION DOCUMENTS LANGCHAIN
# ============================================================
def clean_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'&\w+;', ' ', text)
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

documents = []

for _, row in df_resolved.iterrows():
    content = f"""
Ticket ID: {row['id']}
Objet: {row['objet']}
Description: {row['description']}
Catégorie: {row['categorie']}
Sous-catégorie: {row.get('sous_categorie', '')}
Priorité: {row['priorite']}
Statut: {row['statut']}
""".strip()

    metadata = {
        'ticket_id'      : str(row['id']),
        'categorie'      : str(row['categorie']),
        'sous_categorie' : str(row.get('sous_categorie', '')),
        'priorite'       : str(row['priorite']),
        'statut'         : str(row['statut']),
        'objet'          : str(row['objet'])[:200],
    }

    documents.append(Document(page_content=content, metadata=metadata))

print(f"✅ {len(documents)} documents préparés")
print(f"\nExemple document :")
print(documents[0].page_content)
print(f"\nMétadonnées :")
print(documents[0].metadata)

---
## 5. Modèle d'Embeddings

On utilise **sentence-transformers** avec un modèle
multilingue pour gérer le français correctement.

### Modèle choisi
`paraphrase-multilingual-MiniLM-L12-v2`
- Multilingue (français inclus) ✅
- Léger et rapide ✅
- Bonne qualité sémantique ✅
- Gratuit et local ✅

In [ ]:
# ============================================================
# MODÈLE EMBEDDINGS MULTILINGUE
# ============================================================
print("Chargement du modèle d'embeddings...")
print("(Premier chargement : ~2 minutes)")
print("(Téléchargement du modèle ~120 MB)\n")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Test rapide
test_embedding = embedding_model.embed_query("test réseau wifi")
print(f"✅ Modèle chargé !")
print(f"   Dimension embedding : {len(test_embedding)}")
print(f"   Exemple (5 premières valeurs) : "
      f"{[round(v, 4) for v in test_embedding[:5]]}")

---
## 6. Création de la Base Vectorielle ChromaDB

On indexe tous les documents dans ChromaDB.
Chaque document est transformé en vecteur (embedding)
et stocké pour la recherche par similarité.

In [ ]:
# ============================================================
# CRÉATION CHROMADB
# ============================================================
CHROMA_DIR = '../app/ai_service/chroma_db'
os.makedirs(CHROMA_DIR, exist_ok=True)

print(f"Indexation de {len(documents)} tickets...")
print("(Peut prendre 2-5 minutes...)\n")

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=CHROMA_DIR,
    collection_name="helpdesk_tickets"
)

print(f"✅ ChromaDB créée !")
print(f"   Dossier    : {CHROMA_DIR}")
print(f"   Collection : helpdesk_tickets")
print(f"   Documents  : {vectorstore._collection.count()}")

---
## 7. Test de Recherche par Similarité (collection de base)

Premier test avec la collection de base `helpdesk_tickets`.
La collection `helpdesk_cosine` (optimisée) sera créée à la section suivante.

In [ ]:
# ============================================================
# TEST RECHERCHE AVEC LA COLLECTION DE BASE
# (vectorstore_cosine sera créé dans la section suivante)
# ============================================================
queries = [
    "problème réseau wifi ne fonctionne pas",
    "mot de passe oublié outlook",
    "ordinateur ne démarre pas panne",
    "installation logiciel sage paie",
]

for query in queries:
    print(f"\n{'='*60}")
    print(f"🔍 Query : {query}")
    print(f"{'='*60}")

    # Recherche directe sur vectorstore (collection de base)
    results = vectorstore.similarity_search_with_score(query=query, k=3)

    for i, (doc, score) in enumerate(results):
        similarity = 1 - score
        print(f"  📄 Résultat {i+1} (similarité: {similarity:.4f})")
        print(f"     Ticket    : {doc.metadata['ticket_id']}")
        print(f"     Catégorie : {doc.metadata['categorie']}")
        print(f"     Objet     : {doc.metadata['objet'][:60]}")

In [ ]:
# ============================================================
# CRÉATION COLLECTION COSINUS (similarité cosinus optimisée)
# La collection "helpdesk_cosine" utilise la distance cosinus
# → meilleure mesure pour comparer des embeddings textuels
# ============================================================
import chromadb

client = chromadb.PersistentClient(path=CHROMA_DIR)

print("Collections existantes :")
for col in client.list_collections():
    print(f"  {col.name} : {col.count()} docs")

# Supprimer l'ancienne collection cosine si elle existe
try:
    client.delete_collection("helpdesk_cosine")
    print("✅ Ancienne collection supprimée")
except Exception:
    pass

# Créer la collection cosine avec les documents
vectorstore_cosine = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory=CHROMA_DIR,
    collection_name="helpdesk_cosine",
    collection_metadata={"hnsw:space": "cosine"}
)

count = vectorstore_cosine._collection.count()
print(f"✅ Collection cosine créée : {count} docs")

assert count == len(documents), \
    f"❌ Doublons détectés ! {count} != {len(documents)}"
print("✅ Pas de doublons !")

# ── Définition de la fonction de recherche principale ────────
def search_similar_tickets(query, k=3, categorie=None):
    """
    Rechercher les tickets similaires à une question.

    Args:
        query     : texte de la question / problème
        k         : nombre de résultats à retourner
        categorie : filtre optionnel par catégorie

    Returns:
        list[(Document, score)] : tickets similaires avec scores
    """
    filter_dict = {"categorie": categorie} if categorie else None
    return vectorstore_cosine.similarity_search_with_score(
        query=query, k=k, filter=filter_dict
    )

# Test rapide
test = search_similar_tickets("problème réseau wifi", k=3)
print(f"\nTest recherche cosine :")
for doc, score in test:
    print(f"  sim={1-score:.3f} | {doc.metadata['objet'][:55]}")

---
## 8. Collection Cosinus + Fonction de Recherche

On crée une collection ChromaDB avec **similarité cosinus** — plus adaptée
aux embeddings textuels que la distance euclidienne par défaut.

On définit également la fonction `search_similar_tickets()` réutilisable
dans tout le notebook et dans le backend FastAPI.

In [ ]:
# ============================================================
# SYSTÈME DE SUGGESTIONS RAG
# Approche : extraction intelligente basée sur la similarité
# → pas de LLM externe nécessaire (100% local et gratuit)
# ============================================================

def generate_rag_suggestion(objet, description="", categorie="", k=3):
    """
    Générer une suggestion de réponse basée sur les tickets similaires.

    Principe RAG :
    1. Chercher les tickets passés les plus similaires dans ChromaDB
    2. Extraire les informations utiles (catégorie, objet, solution)
    3. Retourner les suggestions triées par similarité

    Args:
        objet       : objet du ticket
        description : description du ticket
        categorie   : catégorie prédite (filtre optionnel)
        k           : nombre de tickets similaires à récupérer

    Returns:
        dict : suggestions, tickets similaires, confiance RAG
    """
    query = f"{objet} {description}".strip()

    # Recherche des tickets similaires
    results = search_similar_tickets(
        query, k=k,
        categorie=categorie if categorie else None
    )

    if not results:
        return {
            'suggestions'        : [],
            'tickets_similaires' : [],
            'message'            : "Aucun ticket similaire trouvé",
            'confidence'         : 0.0
        }

    tickets_similaires = []
    suggestions        = []

    for doc, score in results:
        similarity = round(1 - score, 4)

        ticket_info = {
            'ticket_id'      : doc.metadata['ticket_id'],
            'objet'          : doc.metadata['objet'],
            'categorie'      : doc.metadata['categorie'],
            'sous_categorie' : doc.metadata.get('sous_categorie', ''),
            'priorite'       : doc.metadata['priorite'],
            'similarity'     : similarity,
        }
        tickets_similaires.append(ticket_info)

        # Générer une suggestion si similarité suffisante
        if similarity > 0.5:
            suggestions.append({
                'ticket_ref' : doc.metadata['ticket_id'],
                'suggestion' : f"Ticket similaire résolu : {doc.metadata['objet'][:80]}",
                'similarity' : similarity
            })

    # Score de confiance RAG = moyenne des 3 meilleures similarités
    confidence = np.mean([1 - score for _, score in results[:3]])

    return {
        'suggestions'        : suggestions,
        'tickets_similaires' : tickets_similaires,
        'confidence'         : round(float(confidence), 4),
        'nb_similaires'      : len(tickets_similaires)
    }


# ============================================================
# TESTS SUGGESTIONS RAG
# ============================================================
test_cases = [
    {'objet': "Problème connexion réseau",    'description': "Impossible de se connecter au wifi",         'categorie': "Réseau"},
    {'objet': "Réinitialisation mot de passe",'description': "Compte bloqué après plusieurs tentatives",   'categorie': "Accès & Compte"},
    {'objet': "PC ne démarre plus",           'description': "Ordinateur éteint ne s allume plus",         'categorie': "Matériel"},
    {'objet': "Création compte email",        'description': "Nouvel employé besoin compte messagerie",    'categorie': "Email & Messagerie"},
]

for tc in test_cases:
    print(f"\n{'='*65}")
    print(f"📋 Ticket    : {tc['objet']}")
    print(f"   Catégorie : {tc['categorie']}")
    print(f"{'='*65}")

    result = generate_rag_suggestion(tc['objet'], tc['description'], tc['categorie'])

    print(f"   Confiance RAG : {result['confidence']:.4f}")
    print(f"   Nb similaires : {result['nb_similaires']}")

    if result['tickets_similaires']:
        print(f"\n   Top 3 tickets similaires :")
        for t in result['tickets_similaires'][:3]:
            print(f"   → [{t['ticket_id']}] sim={t['similarity']:.3f} "
                  f"| {t['categorie']} | {t['objet'][:50]}")

---
## 9. Chargement ChromaDB existante

Fonction pour recharger la base depuis le disque
sans recréer les embeddings.

In [ ]:
# ============================================================
# RECHARGEMENT CHROMADB DEPUIS DISQUE
# Utile pour recharger la base sans recréer les embeddings
# → utilisé par le backend FastAPI au démarrage
# ============================================================
def load_vectorstore(chroma_dir=CHROMA_DIR):
    """Charger la base ChromaDB cosine depuis le disque."""
    return Chroma(
        persist_directory=chroma_dir,
        embedding_function=embedding_model,
        collection_name="helpdesk_cosine"
    )

vs_loaded = load_vectorstore()
count     = vs_loaded._collection.count()

print(f"✅ ChromaDB rechargée depuis disque")
print(f"   Dossier    : {CHROMA_DIR}")
print(f"   Collection : helpdesk_cosine")
print(f"   Documents  : {count}")

assert count == len(documents), f"❌ Nombre incorrect : {count} != {len(documents)}"
print("✅ Intégrité vérifiée — pas de doublons")

# Test de recherche
test_results = vs_loaded.similarity_search("problème imprimante", k=2)
print(f"\nTest requête 'problème imprimante' :")
for doc in test_results:
    print(f"  → [{doc.metadata['ticket_id']}] {doc.metadata['objet'][:60]}")

---
## 10. Sauvegarde Configuration

In [ ]:
# ============================================================
# SAUVEGARDE CONFIG RAG
# ============================================================
import json

config = {
    'embedding_model'      : 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    'chroma_dir'           : CHROMA_DIR,
    'collection_name'      : 'helpdesk_cosine',
    'nb_documents'         : len(documents),
    'similarity_threshold' : 0.5,
    'top_k'                : 3
}

config_path = '../app/ai_service/rag_config.json'
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print("✅ Configuration RAG sauvegardée :")
print(json.dumps(config, indent=2))

---
## 11. Résumé Final

In [ ]:
# ============================================================
# RÉSUMÉ FINAL
# ============================================================
print("=" * 60)
print("  RÉSUMÉ — RAG PIPELINE HELPDESK IA GEISER")
print("=" * 60)

print(f"\n📊 BASE VECTORIELLE")
print(f"   Moteur          : ChromaDB")
print(f"   Documents       : {len(documents)} tickets résolus")
print(f"   Embedding model : multilingual-MiniLM-L12-v2")
print(f"   Dimension       : 384")
print(f"   Stockage        : {CHROMA_DIR}")

print(f"\n📊 PERFORMANCES")
print(f"   Recherche       : similarité cosinus")
print(f"   Top-K           : 3 résultats")
print(f"   Seuil confiance : 0.5")

print(f"\n✅ Fichiers créés :")
print(f"   chroma_db/      → base vectorielle")
print(f"   rag_config.json → configuration")

print(f"\n📋 RÉCAPITULATIF NOTEBOOKS")
print(f"   ✅ 01 — Exploration + Labellisation (1915 tickets, 10 catégories)")
print(f"   ✅ 02 — Classification ML (catégorie 83.8%, priorité 90.6%)")
print(f"   ✅ 03 — Routage agent (65.8%, 2 approches)")
print(f"   ✅ 04 — RAG Pipeline ChromaDB ({len(documents)} docs)")

print(f"\n🚀 Prochaine étape :")
print(f"   Notebook 05 — Analyse Prédictive")
print("=" * 60)